# Computation of the hCF for the Hansen-Zuazua system

The system equations read

$$
\ddot x_1(z,t) = x_1^{''}(z,t), \quad
\ddot x_2(z,t) = x_2^{''}(z,t)
$$

with the boundary conditions

$$
x_1(0,t) = x_2(0,t) = \xi(t),\quad
M \ddot x_1(0,t) = -x_1'(0,t) + x_2'(0,t)
$$

as well as


$$
x_1'(-l_1,t) = -u_1(t),\quad
x_2'(l_2,t) = u_2(t).
$$


## Input parameterisation

In [1]:
import sympy as sp

# symbols
(l1, l2, mass, z, t, tau) = sp.symbols("l_1 l_2 M z t tau")
y1, y2, u1, u2 = [func(t) for func in sp.symbols("y_1 y_2 u_1 u_2", cls=sp.Function)]
# operators
d_sym, s_sym = sp.symbols("d_t sigma", positive=True)

# shorthands
a1 = b1 = d_sym**2 + 2*d_sym
a2 = b2 = d_sym**2 - 2*d_sym
u_vec = sp.Matrix([u1, u2])
y_vec = sp.Matrix([y1, y2])

# general IO matrix
g_mat = sp.Rational(1,4) * sp.Matrix([
    [a1 * s_sym**l1 + a2 * s_sym**(-l1), -s_sym**l1 -s_sym**(-l1)], 
    [b1 * s_sym**l2 + b2 * s_sym**(-l2), s_sym**l2 +s_sym**(-l2)]])

# substitute actual values
g_mat = g_mat.subs([(l1, sp.pi), (l2, 10), (mass, 1)])

print("Parameterisation of the systen input by the flat output:")
display(sp.Eq(u_vec, sp.MatMul(g_mat, y_vec)))

Parameterisation of the systen input by the flat output:


Eq(Matrix([
[u_1(t)],
[u_2(t)]]), Matrix([
[sigma**pi*(d_t**2 + 2*d_t)/4 + (d_t**2 - 2*d_t)/(4*sigma**pi), -sigma**pi/4 - 1/(4*sigma**pi)],
[sigma**10*(d_t**2 + 2*d_t)/4 + (d_t**2 - 2*d_t)/(4*sigma**10),  sigma**10/4 + 1/(4*sigma**10)]])*Matrix([
[y_1(t)],
[y_2(t)]]))

## System analysis

We use the leading column coefficient matrices to get an overview.
As can be seen, the shifts in the second row of the first column are to big.

In [2]:
# from chimera.utils import (
#     get_coeffs, get_matrix_coeffs, show_coeffs, show_mat_coeffs, is_col_reduced,
#     get_column_degrees, get_lccm)
# from chimera.algos import reduce_matrix_shifts
# from chimera.genpoly import GenPoly

In [3]:
import chimera as ch

print("Leading coefficient matrices:")
print("LCCM for derivatives:")
display(ch.get_lccm(g_mat, d_sym))
print("LCCM for shifts:")
display(ch.get_lccm(g_mat, s_sym))
print("LCCM for derivatives and shifts :")
lccm = ch.get_lccm(ch.get_lccm(g_mat, d_sym), s_sym)
display(lccm)
print("Is G column-reduced w.r.t. shifts?")
ch.is_col_reduced(g_mat, s_sym)

Leading coefficient matrices:
LCCM for derivatives:


Matrix([
[(sigma**(2*pi) + 1)/(4*sigma**pi), (-sigma**(2*pi) - 1)/(4*sigma**pi)],
[    (sigma**20 + 1)/(4*sigma**10),      (sigma**20 + 1)/(4*sigma**10)]])

LCCM for shifts:


Matrix([
[               0,   0],
[d_t**2/4 + d_t/2, 1/4]])

LCCM for derivatives and shifts :


Matrix([
[  0,   0],
[1/4, 1/4]])

Is G column-reduced w.r.t. shifts?


False

## Shift reduction

We employ the developed algorithm to reduce the shifts


In [4]:
gb_mat, l_mat = ch.reduce_matrix_shifts(g_mat, d_sym, s_sym)

print("Computed transformation:")
display(l_mat)
print("Transformed matrix:")
display(gb_mat)

Reducing row 1
Reducing col 0
Computed transformation:


Matrix([
[                                                                                                                                                                  1, 0],
[-sigma**(-10 + pi) + sigma**(-10 + 3*pi)*(d_t + 2)/(d_t - 2) + sigma**(10 - 3*pi)*(d_t - 2)/(d_t + 2) - sigma**(10 - pi) - 16*exp(-20 + 6*pi)/((d_t - 2)*(d_t + 2)), 1]])

Transformed matrix:


Matrix([
[                                                                                                                                                                                                                                        sigma**pi*(d_t**2 + 2*d_t)/4 + (d_t**2 - 2*d_t)/(4*sigma**pi),                                                                                                                                                                                                         -sigma**pi/4 - 1/(4*sigma**pi)],
[sigma**10*(d_t**2 + 2*d_t)/4 + (sigma**pi*(d_t**2 + 2*d_t)/4 + (d_t**2 - 2*d_t)/(4*sigma**pi))*(-sigma**(-10 + pi) + sigma**(-10 + 3*pi)*(d_t + 2)/(d_t - 2) + sigma**(10 - 3*pi)*(d_t - 2)/(d_t + 2) - sigma**(10 - pi) - 16*exp(-20 + 6*pi)/((d_t - 2)*(d_t + 2))) + (d_t**2 - 2*d_t)/(4*sigma**10), sigma**10/4 + (-sigma**pi/4 - 1/(4*sigma**pi))*(-sigma**(-10 + pi) + sigma**(-10 + 3*pi)*(d_t + 2)/(d_t - 2) + sigma**(10 - 3*pi)*(d_t - 2)/(d_t + 2) - sigma**(10 - p

In [5]:
print("Leading coefficient matrices:")
print("LCCM for derivatives:")
display(ch.get_lccm(gb_mat, d_sym))
print("LCCM for shifts:")
display(ch.get_lccm(gb_mat, s_sym))
print("LCCM for derivatives and shifts :")
lccm = ch.get_lccm(ch.get_lccm(gb_mat, d_sym), s_sym)
display(lccm)
print("Is the transformed G column-reduced w.r.t. derivatives and shifts?")
ch.is_col_reduced(gb_mat, s_sym)


Leading coefficient matrices:
LCCM for derivatives:


Matrix([
[                                                                                                        (sigma**(2*pi) + 1)/(4*sigma**pi),                                                                                                                                                                                                      (-sigma**(2*pi) - 1)/(4*sigma**pi)],
[sigma**(3*pi)*(4*sigma**40*exp(20)*exp(-6*pi)/sigma**(7*pi) + 4*sigma**20*sigma**pi*exp(20)*exp(-6*pi))*exp(-20)*exp(6*pi)/(16*sigma**30), sigma**(3*pi)*(8*sigma**40*exp(20)*exp(-6*pi)/sigma**(3*pi) - 4*sigma**40*exp(20)*exp(-6*pi)/sigma**(7*pi) - 4*sigma**20*sigma**pi*exp(20)*exp(-6*pi) + 8*sigma**20*exp(20)*exp(-6*pi)/sigma**(3*pi))*exp(-20)*exp(6*pi)/(16*sigma**30)]])

LCCM for shifts:


Matrix([
[                 d_t*(d_t + 2)/4,   0],
[-4*d_t*exp(-20 + 6*pi)/(d_t - 2), 1/2]])

LCCM for derivatives and shifts :


Matrix([
[1/4,   0],
[  0, 1/2]])

Is the transformed G column-reduced w.r.t. derivatives and shifts?


True

## Stating the controller form

Substitute terms that will not be members of the state by partial integrations and introduce the new controller form state.

In [6]:
display(gb_mat)
conv_mat = gb_mat.applyfunc(lambda x: ch.inverse_laplace_transform(x, d_sym, s_sym, t))
display(conv_mat)


Matrix([
[                                                                                                                                                                                                                                        sigma**pi*(d_t**2 + 2*d_t)/4 + (d_t**2 - 2*d_t)/(4*sigma**pi),                                                                                                                                                                                                         -sigma**pi/4 - 1/(4*sigma**pi)],
[sigma**10*(d_t**2 + 2*d_t)/4 + (sigma**pi*(d_t**2 + 2*d_t)/4 + (d_t**2 - 2*d_t)/(4*sigma**pi))*(-sigma**(-10 + pi) + sigma**(-10 + 3*pi)*(d_t + 2)/(d_t - 2) + sigma**(10 - 3*pi)*(d_t - 2)/(d_t + 2) - sigma**(10 - pi) - 16*exp(-20 + 6*pi)/((d_t - 2)*(d_t + 2))) + (d_t**2 - 2*d_t)/(4*sigma**10), sigma**10/4 + (-sigma**pi/4 - 1/(4*sigma**pi))*(-sigma**(-10 + pi) + sigma**(-10 + 3*pi)*(d_t + 2)/(d_t - 2) + sigma**(10 - 3*pi)*(d_t - 2)/(d_t + 2) - sigma**(10 - p

Matrix([
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  -DiracDelta(t - pi, 1)/2 + DiracDelta(t - pi, 2)/4 + DiracDelta(t + pi, 1)/2 + DiracDelta(t + pi, 2)/4,                                                                                                                                                                                                                                                                                                                                                                                                                     

In [7]:
res = sp.Integral(conv_mat.subs(t,tau) @ y_vec.subs(t, t-tau), (tau, -sp.oo, sp.oo)).doit()
display(res[0])
display(res[1])


-y_2(t - pi)/4 - y_2(t + pi)/4 - Subs(Derivative(y_1(_xi_1), _xi_1), _xi_1, t - pi)/2 + Subs(Derivative(y_1(_xi_1), _xi_1), _xi_1, t + pi)/2 + Subs(Derivative(y_1(_xi_1), (_xi_1, 2)), _xi_1, t - pi)/4 + Subs(Derivative(y_1(_xi_1), (_xi_1, 2)), _xi_1, t + pi)/4

Integral(-32*y_1(t - tau)*exp(-20)*exp(8*pi)*exp(2*tau)*Heaviside(tau + pi) + 32*y_1(t - tau)*exp(-20)*exp(8*pi)*exp(2*tau)*Heaviside(tau - 10 + 4*pi) - 16*y_1(t - tau)*exp(-20)*exp(6*pi)*DiracDelta(tau - pi) - 16*y_1(t - tau)*exp(-20)*exp(6*pi)*DiracDelta(tau + pi) + 16*y_1(t - tau)*DiracDelta(tau - 10 + 4*pi) + 16*y_1(t - tau)*DiracDelta(tau - 4*pi + 10) + 6*y_1(t - tau)*DiracDelta(tau - 10 + 4*pi, 1) + y_1(t - tau)*DiracDelta(tau - 10 + 4*pi, 2) - 6*y_1(t - tau)*DiracDelta(tau - 4*pi + 10, 1) + y_1(t - tau)*DiracDelta(tau - 4*pi + 10, 2) + 32*y_1(t - tau)*exp(-20)*exp(8*pi)*exp(-2*tau)*Heaviside(tau - pi) - 32*y_1(t - tau)*exp(-20)*exp(8*pi)*exp(-2*tau)*Heaviside(tau - 4*pi + 10) + 4*y_2(t - tau)*exp(-20)*exp(4*pi)*exp(2*tau)*Heaviside(tau - pi) + 4*y_2(t - tau)*exp(-20)*exp(8*pi)*exp(2*tau)*Heaviside(tau + pi) - 4*y_2(t - tau)*exp(-20)*exp(4*pi)*exp(2*tau)*Heaviside(tau - 10 + 2*pi) - 4*y_2(t - tau)*exp(-20)*exp(8*pi)*exp(2*tau)*Heaviside(tau - 10 + 4*pi) + 2*y_2(t - tau)*DiracDelt

In [8]:
y3 = sp.Function("y_3")
print(y3.__class__)
y3t = y3(t)
print(y3t.__class__)

<class 'sympy.core.function.UndefinedFunction'>
y_3


In [9]:
from sympy.core.function import AppliedUndef


def simplify_conv_integrals(expr):
    if isinstance(expr, sp.Integral):
        int_kernel = expr.args[0]
        int_var, lb, ub = expr.args[1]
        if lb == -sp.oo and ub == sp.oo:
            res = 0
            for r in _simplify_conv(int_kernel, int_var, lb, ub):
                res += expr.func(r[0], r[1])
            return res
        else:
            return expr

    # traverse the tree
    new_args = tuple(simplify_conv_integrals(arg) for arg in expr.args)
    if new_args != expr.args:
        try:
            expr = expr.func(*new_args).doit()
        except Exception as e:
            # Fallback: keep original node if reconstruction fails
            print(e)
            print(expr)
            print(expr.args)
            print(new_args)
            pass

    return expr


def _simplify_conv(kernel, var, lower, upper):
    results = []

    # identify applied undefined functions that are being convolved
    c_funcs = kernel.atoms(AppliedUndef)

    # # collect expressions concerning them
    # col_kernel = kernel.collect(c_funcs)
    # display(col_kernel)

    # treat each function separately
    for func in c_funcs:
        factor = kernel.coeff(func)
        res = _simplify_conv_factor(factor, var, lower, upper)
        for r in res:
            results.append((r[0] * func, r[1]))

    return results


def _simplify_conv_factor(factor, var, lower, upper):
    res = []

    # display(factor)

    # start with isolated dirichlet deltas
    dir_factors = 0
    hea_factors = {}
    rem_factors = 0
    for term in sp.Add.make_args(factor.expand()):
        if term.has(sp.DiracDelta) and not term.has(sp.Heaviside):
            dir_factors += term
        elif term.has(sp.Heaviside):
            # display(term)
            f = term.atoms(sp.Heaviside)
            assert len(f) == 1
            f = list(f)[0]
            # display(f)
            c = (term / f).simplify()
            if c in hea_factors:
                hea_factors[c] += [f]
            else:
                if -c in hea_factors:
                    hea_factors[-c] += [-f]
                else:
                    hea_factors[c] = [f]
        else:
            rem_factors += term

    # just re-add the Diracs, sympy will simplify them
    res.append((dir_factors, (var, lower, upper)))

    # continue with heavyside expressions
    for c, funcs in hea_factors.items():
        # order funcs by arg and correct the signs such that
        # expr = factor * (H(lb) - H(ub))

        # extract and order the bounds
        bounds = []
        for f in funcs:
            if not isinstance(f, sp.Heaviside):
                # ignore the minus sign
                if isinstance(f, sp.Mul):
                    for a in f.args:
                        if isinstance(a, sp.Heaviside):
                            f = a
                            break
                else:
                    raise NotImplementedError
            bounds.append(f.args[0].subs(var, 0))
        s_funcs, s_bounds = zip(*sorted(zip(funcs, bounds), key=lambda x:x[1]))

        # check for minus sign at the wrong place
        if not isinstance(s_funcs[0], sp.Heaviside):
            s_funcs = [-1 * f for f in s_funcs]
            c *= -1

        # append new factor with reduced bounds
        res.append((c, (var, *bounds)))


    # add the remainder
    res.append((rem_factors, (var, lower, upper)))
    return res




In [11]:
from chimera.utils import simplify_conv_integrals

conv = res
display(conv)
s_conv = simplify_conv_integrals(conv)
display(s_conv)


Matrix([
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

Matrix([
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             -y_2(t - pi)/4 - y_2(t + pi)/4 - Subs(Derivative(y_1(_xi_1), _xi_1), _xi_1, t - pi)/2 + Subs(Derivative(y_1(_xi_1), _xi_1), _xi_1, t + pi)/2 + Subs(Derivative(y_1(_xi_1), (_xi_1, 2)), _xi_1, t - pi)/4 + Subs(Derivative(y_1(_xi_1), (_xi_1, 2)), _xi_1, t + pi)/4],
[-4*y_1(t - pi)*exp(-20)*exp(6*pi) - 4*y_1(t + pi)*exp(-20

In [ ]:
ys[1]

In [ ]:
for term in sp.Add.make_args(ys[1].expand()):
    if not term.has(sp.Integral):
        continue
    if "y_1" not in str(term.atoms(sp.Function)):
        continue

    display(term)


In [ ]:
# ("y1" in
str(ys[1].atoms(sp.Function))
# )